In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MarketDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

class MarketPredictionNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128, 64, 32], dropout_rate=0.3):
        super(MarketPredictionNN, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
    main_features.extend([f'ME_prod_M*_E*', f'ME_ratio_M*_E*'])
    
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    main_features.extend([f'MI_prod_M*_I*', f'MI_ratio_M*_I*', f'MI_spread_M*_I*'])
    
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    main_features.extend([f'MP_prod_M*_P*', f'MP_ratio_M*_P*'])
    
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    if 'return' in 'M*'.lower():
        data[f'MV_sharpe_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
        main_features.append(f'MV_sharpe_M*_V*')
    main_features.extend([f'MV_ratio_M*_V*', f'MV_prod_M*_V*'])
    
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.extend([f'MS_prod_M*_S*', f'MS_weighted_M*_S*'])
    
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.extend([f'EI_diff_E*_I*', f'EI_ratio_E*_I*', f'EI_prod_E*_I*'])
    
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.extend([f'EP_prod_E*_P*', f'EP_ratio_E*_P*'])
    
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.extend([f'EV_prod_E*_V*', f'EV_uncertainty_E*_V*'])
    
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.extend([f'ES_prod_E*_S*', f'ES_diff_E*_S*'])
    
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.extend([f'IP_prod_I*_P*', f'IP_discount_I*_P*'])
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
    
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
    
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.extend([f'PV_prod_P*_V*', f'PV_stability_P*_V*'])
    
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.extend([f'PS_prod_P*_S*', f'PS_contrarian_P*_S*'])
    
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.extend([f'VS_prod_V*_S*', f'VS_panic_V*_S*'])

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 
    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 
    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 
    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']
    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

def train_model(model, train_loader, val_loader, epochs=100, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)
    
    best_val_loss = float('inf')
    patience = 20
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    model.load_state_dict(torch.load('best_model.pt'))
    return model

# Load and preprocess data
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"]).values
val_x = val_split.drop(columns=["target"]).values
train_y = train_split['target'].values
val_y = val_split['target'].values

# Scale the features
scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)
val_x_scaled = scaler.transform(val_x)

# Save scaler for inference
joblib.dump(scaler, 'scaler.pkl')

# Create datasets and dataloaders
train_dataset = MarketDataset(train_x_scaled, train_y)
val_dataset = MarketDataset(val_x_scaled, val_y)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Initialize and train model
input_dim = train_x_scaled.shape[1]
print(f'Neural Network TRAINING with {input_dim} features...')

model = MarketPredictionNN(input_dim=input_dim).to(device)
model = train_model(model, train_loader, val_loader, epochs=200, lr=0.001)

# Save the model
torch.save(model.state_dict(), 'pytorch_model.pt')
print('Model training complete!')

# Load model and scaler for inference
loaded_model = MarketPredictionNN(input_dim=input_dim).to(device)
loaded_model.load_state_dict(torch.load('pytorch_model.pt'))
loaded_model.eval()
loaded_scaler = joblib.load('scaler.pkl')

SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        test_processed = preprocessing(test_df, 'inference')
        
        # Scale the test data
        test_scaled = loaded_scaler.transform(test_processed.values)
        
        # Convert to tensor and predict
        test_tensor = torch.FloatTensor(test_scaled).to(device)
        
        with torch.no_grad():
            prediction = loaded_model(test_tensor).cpu().numpy()[0][0]
        
        signal = convert_ret_to_signal(prediction)
        print(float(signal))
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Neural Network TRAINING with 86 features...
Epoch [10/200], Train Loss: 0.004160, Val Loss: 0.000248
Epoch [20/200], Train Loss: 0.000790, Val Loss: 0.000127
Epoch [30/200], Train Loss: 0.000247, Val Loss: 0.000106
Epoch [40/200], Train Loss: 0.000178, Val Loss: 0.000104
Epoch [50/200], Train Loss: 0.000147, Val Loss: 0.000102
Early stopping at epoch 54
Model training complete!
1.2888979855924845
1.0190757680684328
1.5159825552254915
1.4388286266475916
0.9084352049976587
1.268750311806798
0.8553676996380091
1.1565227750688791
1.3727725241333246
1.2243001479655504
